In [1]:
import import_ipynb
from Codigos import *

from pathlib import Path 
import json 
import joblib 
import numpy as np 
import pandas as pd 
import tensorflow as tf

In [2]:
# ============================================================
# CONFIGURACIÓN DE RUTAS
# ============================================================
from pathlib import Path

# Directorio de trabajo actual del notebook
BASE_DIR = Path.cwd()

DIR_ARTEFACTOS = BASE_DIR / "modelo_lpips_exportado"
DIR_RESULTADOS = BASE_DIR / "resultados_optimizacion"

RUTA_PREPROCESAMIENTO = (
    DIR_ARTEFACTOS / "preprocesamiento.joblib"
)
RUTA_MODELO = (
    DIR_RESULTADOS / "modelo_lpips_optimizado.keras"
)
RUTA_CONFIGURACION = (
    DIR_RESULTADOS / "configuracion_modelo.json"
)

print("Directorio de trabajo:", BASE_DIR)
print("Preprocesamiento:", RUTA_PREPROCESAMIENTO)
print("Modelo:", RUTA_MODELO)
print("Configuración:", RUTA_CONFIGURACION)

RUTA_PREPROCESAMIENTO = DIR_ARTEFACTOS / "preprocesamiento.joblib"
RUTA_MODELO = DIR_RESULTADOS / "modelo_lpips_optimizado.keras"
RUTA_CONFIGURACION = DIR_RESULTADOS / "configuracion_modelo.json"




Directorio de trabajo: d:\TrabajoGrado\Codigos\Funciones_Matematicas
Preprocesamiento: d:\TrabajoGrado\Codigos\Funciones_Matematicas\modelo_lpips_exportado\preprocesamiento.joblib
Modelo: d:\TrabajoGrado\Codigos\Funciones_Matematicas\resultados_optimizacion\modelo_lpips_optimizado.keras
Configuración: d:\TrabajoGrado\Codigos\Funciones_Matematicas\resultados_optimizacion\configuracion_modelo.json


In [3]:
def extraer_features(path, prefix):
    f = {}

    f[f"{prefix}_media_lum"] = obtener_media_luminancia(path)
    f[f"{prefix}_var_lum"] = obtener_varianza_luminancia(path)
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)


    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    f[f"{prefix}_grad_mean"] = obtener_contraste_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_glcm(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_glcm(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    f[f"{prefix}_entropia_shannon"] = obtener_entropia_shannon(path)
    f[f"{prefix}_entropia_espectral"] = obtener_entropia_espectral(path)
    f[f"{prefix}_entropia_gradiente"] = obtener_entropia_gradiente(path)
    f[f"{prefix}_entropia_glcm"] = obtener_entropia_glcm(path)

    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)
    f[f"{prefix}_hue_var"] = obtener_varianza_matiz(path)

    f[f"{prefix}_megapixeles"] = obtener_megapixeles(path)
    f[f"{prefix}_aspect_ratio"] = obtener_aspect_ratio(path)

    return f

In [4]:
# ============================================================
# CARGAR ARTEFACTOS UNA SOLA VEZ
# ============================================================

def cargar_artefactos():
    rutas = [
        RUTA_PREPROCESAMIENTO,
        RUTA_MODELO,
        RUTA_CONFIGURACION,
    ]

    faltantes = [str(ruta) for ruta in rutas if not ruta.exists()]

    if faltantes:
        raise FileNotFoundError(
            "No se encontraron los siguientes archivos:\n"
            + "\n".join(faltantes)
        )

    preprocesamiento = joblib.load(RUTA_PREPROCESAMIENTO)
    modelo = tf.keras.models.load_model(RUTA_MODELO)

    with open(RUTA_CONFIGURACION, "r", encoding="utf-8") as archivo:
        configuracion = json.load(archivo)

    return modelo, preprocesamiento, configuracion


MODELO, PREPROCESAMIENTO, CONFIGURACION = cargar_artefactos()

In [5]:
# ============================================================
# PREPARAR UNA IMAGEN PARA LA INFERENCIA
# ============================================================

def preparar_entrada(
    path_imagen,
    formato_salida,
    metodo,
    calidad,
):
    """
    Construye las variables de entrada usando las mismas reglas
    aplicadas durante la preparación del conjunto de entrenamiento.

    No recodifica la imagen: estima LPIPS antes de hacerlo.
    """

    path_imagen = Path(path_imagen)

    if not path_imagen.exists():
        raise FileNotFoundError(
            f"No se encontró la imagen: {path_imagen}"
        )

    if not path_imagen.is_file():
        raise ValueError("La ruta indicada no corresponde a un archivo.")

    # --------------------------------------------------------
    # 1. Detectar el formato original desde la extensión
    # --------------------------------------------------------

    formato_entrada = path_imagen.suffix.lower().lstrip(".")

    if formato_entrada == "jpeg":
        formato_entrada = "jpg"

    formatos_validos = {"jpg", "png", "webp"}

    if formato_entrada not in formatos_validos:
        raise ValueError(
            f"Formato de entrada no admitido: {formato_entrada}. "
            f"Formatos admitidos: {sorted(formatos_validos)}"
        )

    formato_salida = formato_salida.lower().strip().lstrip(".")

    if formato_salida == "jpeg":
        formato_salida = "jpg"

    if formato_salida not in formatos_validos:
        raise ValueError(
            f"Formato de salida no admitido: {formato_salida}"
        )

    if not isinstance(calidad, (int, np.integer)):
        raise TypeError("La calidad debe ser un número entero.")

    if calidad not in {10, 30, 50, 70, 90}:
        raise ValueError(
            "La calidad debe ser uno de los niveles "
            "experimentales: 10, 30, 50, 70 o 90."
        )

    # --------------------------------------------------------
    # 2. Extraer las características de la imagen original
    # --------------------------------------------------------

    caracteristicas = extraer_features(
        str(path_imagen),
        prefix="orig",
    )

    X = pd.DataFrame([caracteristicas])

    # --------------------------------------------------------
    # 3. Aplicar las mismas eliminaciones del entrenamiento
    # --------------------------------------------------------

    columnas_eliminar = [
        "orig_var_lum",
        "orig_edges_s1",
        "orig_edges_s2",
        "orig_hlfr",
        "orig_rango",
    ]

    X = X.drop(
        columns=columnas_eliminar,
        errors="ignore",
    )

    # --------------------------------------------------------
    # 4. Tratar hue_var como en el entrenamiento
    # --------------------------------------------------------

    if "orig_hue_var" not in X.columns:
        raise ValueError(
            "La extracción no generó la característica orig_hue_var."
        )

    hue_faltante = X["orig_hue_var"].isna().astype(int)

    X["orig_hue_var_faltante"] = hue_faltante

    mediana_hue = PREPROCESAMIENTO["mediana_hue"]

    X["orig_hue_var"] = X["orig_hue_var"].fillna(
        mediana_hue
    )

    # --------------------------------------------------------
    # 5. Añadir los parámetros de recodificación
    # --------------------------------------------------------

    X["formato_entrada"] = formato_entrada
    X["formato_salida"] = formato_salida
    X["metodo"] = metodo
    X["calidad"] = calidad

    columnas_categoricas = PREPROCESAMIENTO[
        "columnas_categoricas"
    ]

    columnas_modelo = PREPROCESAMIENTO["columnas_modelo"]

    # Verificar que las categorías seleccionadas existan
    # en el esquema usado durante el entrenamiento.
    for columna in columnas_categoricas:
        valor = X[columna].iloc[0]
        prefijo = f"{columna}_"

        categorias_conocidas = {
            nombre[len(prefijo):]
            for nombre in columnas_modelo
            if nombre.startswith(prefijo)
        }

        if valor not in categorias_conocidas:
            raise ValueError(
                f"Categoría no reconocida para '{columna}': "
                f"'{valor}'. Categorías disponibles: "
                f"{sorted(categorias_conocidas)}"
            )

    # --------------------------------------------------------
    # 6. Codificación One-Hot y alineación de columnas
    # --------------------------------------------------------

    X = pd.get_dummies(
        X,
        columns=columnas_categoricas,
        dtype=int,
    )

    # Reproduce el esquema de columnas del entrenamiento.
    # Las columnas ausentes se rellenan con cero.
    X = X.reindex(
        columns=columnas_modelo,
        fill_value=0,
    )

    # --------------------------------------------------------
    # 7. Normalizar exclusivamente variables continuas
    # --------------------------------------------------------

    columnas_continuas = PREPROCESAMIENTO[
        "columnas_continuas"
    ]

    X.loc[:, columnas_continuas] = (
        PREPROCESAMIENTO["scaler"].transform(
            X[columnas_continuas]
        )
    )

    # Comprobaciones finales
    if X.isna().any().any():
        columnas_nulas = X.columns[X.isna().any()].tolist()
        raise ValueError(
            f"Quedaron valores faltantes en la entrada: {columnas_nulas}"
        )

    if X.columns.tolist() != columnas_modelo:
        raise ValueError(
            "Las columnas no coinciden con las del entrenamiento."
        )

    return X

In [6]:
# ============================================================
# PREDICCIÓN ANTICIPADA DE LPIPS
# ============================================================

def predecir_lpips(
    path_imagen,
    formato_salida,
    metodo,
    calidad,
):
    """
    Estima LPIPS antes de realizar la recodificación.

    Devuelve tanto la predicción original del modelo como
    una estimación limitada inferiormente a cero para su
    interpretación dentro del dominio de LPIPS.
    """

    X = preparar_entrada(
        path_imagen=path_imagen,
        formato_salida=formato_salida,
        metodo=metodo,
        calidad=calidad,
    )

    prediccion_original = float(
        MODELO.predict(X, verbose=0).reshape(-1)[0]
    )

    prediccion_no_negativa = max(0.0, prediccion_original)

    return {
        "lpips_estimado_original": prediccion_original,
        "lpips_estimado_no_negativo": prediccion_no_negativa,
        "prediccion_negativa": prediccion_original < 0,
        "formato_entrada": Path(path_imagen).suffix.lower().lstrip("."),
        "formato_salida": formato_salida,
        "metodo": metodo,
        "calidad": calidad,
    }




In [14]:
# ============================================================
# EJEMPLO DE USO
# ============================================================

resultado = predecir_lpips(
        path_imagen="Imagenes/jpg/apparel (1) (Copy).jpg",
        formato_salida="webp",
        metodo="pillow",
        calidad=50,
    )

print("\nRESULTADO DE LA PREDICCIÓN")
print("=" * 45)

for clave, valor in resultado.items():
    print(f"{clave}: {valor}")


RESULTADO DE LA PREDICCIÓN
lpips_estimado_original: 0.02168445661664009
lpips_estimado_no_negativo: 0.02168445661664009
prediccion_negativa: False
formato_entrada: jpg
formato_salida: webp
metodo: pillow
calidad: 50


C:\Users\santi\AppData\Local\Temp\ipykernel_12116\163566665.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.00940817]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:, columnas_continuas] = (


In [8]:
from PIL import Image

def verificar_rgb(ruta_imagen: str) -> bool:
    """
    Verifica si una imagen está en modo RGB.

    Retorna:
        True: si la imagen está en modo RGB.
        False: si está en otro modo, como RGBA, L, P o CMYK.
    """
    with Image.open(ruta_imagen) as imagen:
        return imagen.mode == "RGB"

In [9]:
print(verificar_rgb("Imagenes/jpg/apparel (1) (Copy).jpg"))

True
